# Attention - advanced walkthrough

Run every cell from top to bottom. The notebook prints intermediate values and draws visualizations so the math stays visible.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

np.random.seed(7)
plt.style.use('default')

## 1. Build query, key, and value matrices
Each row is one token. Attention compares each query to all keys, then blends the values.

In [ ]:
tokens = ['I', 'like', 'math']
Q = np.array([[1.0, 0.0], [0.7, 0.7], [0.0, 1.0]])
K = np.array([[1.0, 0.1], [0.6, 0.8], [0.1, 1.0]])
V = np.array([[1.0, 0.0, 0.2], [0.2, 1.0, 0.2], [0.0, 0.3, 1.0]])
print('Q shape:', Q.shape)
print('K shape:', K.shape)
print('V shape:', V.shape)
display(pd.DataFrame(Q, index=tokens, columns=['q1', 'q2']))

## 2. Compute scaled dot-product scores

In [ ]:
d_k = Q.shape[1]
scores = Q @ K.T / np.sqrt(d_k)
score_df = pd.DataFrame(scores, index=tokens, columns=tokens)
print('scores = Q @ K.T / sqrt(d_k)')
display(score_df)

## 3. Softmax turns scores into attention weights

In [ ]:
def softmax(x, axis=-1):
    shifted = x - np.max(x, axis=axis, keepdims=True)
    exp = np.exp(shifted)
    return exp / exp.sum(axis=axis, keepdims=True)

weights = softmax(scores, axis=1)
weights_df = pd.DataFrame(weights, index=tokens, columns=tokens)
print('Each row sums to:', weights.sum(axis=1))
display(weights_df)

## 4. Visualize attention and compute context vectors

In [ ]:
context = weights @ V
context_df = pd.DataFrame(context, index=tokens, columns=['v1', 'v2', 'v3'])
print('context = attention_weights @ V')
display(context_df)

fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(weights, cmap='Blues', vmin=0, vmax=1)
ax.set_xticks(range(len(tokens)))
ax.set_yticks(range(len(tokens)))
ax.set_xticklabels(tokens)
ax.set_yticklabels(tokens)
ax.set_xlabel('key token')
ax.set_ylabel('query token')
ax.set_title('Attention weights')
for i in range(len(tokens)):
    for j in range(len(tokens)):
        ax.text(j, i, f'{weights[i, j]:.2f}', ha='center', va='center')
fig.colorbar(im, ax=ax)
plt.show()

## 5. Add a causal mask
A decoder cannot attend to future tokens, so future scores are set to a very negative value before softmax.

In [ ]:
mask = np.triu(np.ones_like(scores), k=1).astype(bool)
masked_scores = scores.copy()
masked_scores[mask] = -1e9
masked_weights = softmax(masked_scores, axis=1)
print('Causal attention weights:')
display(pd.DataFrame(masked_weights, index=tokens, columns=tokens))
fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(masked_weights, cmap='Oranges', vmin=0, vmax=1)
ax.set_xticks(range(len(tokens)))
ax.set_yticks(range(len(tokens)))
ax.set_xticklabels(tokens)
ax.set_yticklabels(tokens)
ax.set_title('Causal masked attention')
for i in range(len(tokens)):
    for j in range(len(tokens)):
        ax.text(j, i, f'{masked_weights[i, j]:.2f}', ha='center', va='center')
fig.colorbar(im, ax=ax)
plt.show()

Try changing the query vector for `math` and watch its attention row change.